In [1]:
%load_ext autoreload
%autoreload 2
import os
os.chdir('/workspace/RL4Auditing')
import numpy as np
import matplotlib.pyplot as plt

In [4]:
# import numpy as np
# from sklearn.model_selection import train_test_split

# # Load the data
# x_data = np.load('../vu2/RL4Auditing/x_data.npy')
# y_data = np.load('../vu2/RL4Auditing/y_data.npy')

# # Split the data into training and remaining (test + validation) sets
# x_train, x_temp, y_train, y_temp = train_test_split(
#     x_data, y_data, test_size=0.4, random_state=42
# )

# # Further split the remaining data into validation and test sets without overlap
# split_index = len(x_temp) // 2
# x_val, x_test = x_temp[:split_index], x_temp[split_index:]
# y_val, y_test = y_temp[:split_index], y_temp[split_index:]

# # Print the shapes of the splits
# print("Training set:", x_train.shape, y_train.shape)
# print("Validation set:", x_val.shape, y_val.shape)
# print("Test set:", x_test.shape, y_test.shape)

# # Save the splits to separate .npz files
# np.savez('./split_data/train.npz', x=x_train, y=y_train)
# np.savez('./split_data/val.npz', x=x_val, y=y_val)
# np.savez('./split_data/test.npz', x=x_test, y=y_test)


Training set: (36000, 300, 300, 3) (36000, 15)
Validation set: (12000, 300, 300, 3) (12000, 15)
Test set: (12000, 300, 300, 3) (12000, 15)


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import numpy as np

# Custom CNN Model
class CNN5Layer(nn.Module):
    def __init__(self, num_classes=15):
        super(CNN5Layer, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)  # Layer 1
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)  # Layer 2
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)  # Layer 3
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)  # Layer 4
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1)  # Layer 5
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # Downsampling
        self.relu = nn.ReLU()  # Activation function
        
        self.fc = nn.Linear(512 * 7 * 7, num_classes)  # Fully Connected Layer
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = self.pool(self.relu(self.conv4(x)))
        x = self.pool(self.relu(self.conv5(x)))
        
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)  # No activation (BCEWithLogitsLoss expects raw logits)
        return x


# Custom Dataset Class
class NPZImageDataset(torch.utils.data.Dataset):
    def __init__(self, npz_path, transform=None):
        data = np.load(npz_path)
        self.images = data['x']
        self.labels = data['y']  # Multi-labels as a numpy array
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)  # Use float for BCE loss
        
        if len(image.shape) == 2:  # Grayscale to 3 channels
            image = image.unsqueeze(0).repeat(3, 1, 1)
        else:
            image = image.permute(2, 0, 1)  # (H, W, C) -> (C, H, W)

        if self.transform:
            image = self.transform(image)
        return image, label


# Training Code
def train_cnn(train_loader, test_loader, num_classes=15, num_epochs=10, learning_rate=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = CNN5Layer(num_classes=num_classes).to(device)
    criterion = nn.BCEWithLogitsLoss()  # For multi-label classification
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    print("Starting Training...")
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {train_loss/len(train_loader):.4f}")

        # Evaluation
        model.eval()
        total_loss = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                total_loss += loss.item()
        print(f"Test Loss: {total_loss/len(test_loader):.4f}\n")

    print("Training Completed!")
    return model


In [8]:
TRAIN_NPZ = "../vu2/RL4Auditing/split_data/train.npz"
TEST_NPZ = "../vu2/RL4Auditing/split_data/test.npz"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Data Loaders
train_dataset = NPZImageDataset(TRAIN_NPZ, transform=transform)
test_dataset = NPZImageDataset(TEST_NPZ, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Train the model
model = train_cnn(train_loader, test_loader, num_classes=15, num_epochs=10, learning_rate=0.001)

Starting Training...
Epoch [1/10], Loss: 0.5019
Test Loss: 0.3015

Epoch [2/10], Loss: 0.2744
Test Loss: 0.2376

Epoch [3/10], Loss: 0.2423
Test Loss: 0.2170

Epoch [4/10], Loss: 0.2260
Test Loss: 0.2032

Epoch [5/10], Loss: 0.2145
Test Loss: 0.1980

Epoch [6/10], Loss: 0.2066
Test Loss: 0.1901

Epoch [7/10], Loss: 0.2000
Test Loss: 0.1846

Epoch [8/10], Loss: 0.1957
Test Loss: 0.1809

Epoch [9/10], Loss: 0.1917
Test Loss: 0.1795

Epoch [10/10], Loss: 0.1881
Test Loss: 0.1737

Training Completed!


In [11]:
# save_path = '../vu2/RL4Auditing/split_data/cnn5layer_multi_label.pth'
# torch.save(model.state_dict(), save_path)
# print(f"Model saved as {save_path}")

Model saved as ../vu2/RL4Auditing/split_data/cnn5layer_multi_label.pth


In [12]:
def evaluate_per_class_accuracy(model, test_loader, num_classes=15, threshold=0.5):
    """
    Evaluate per-class accuracy for a multi-label classification model.
    
    Parameters:
    - model (nn.Module): Trained PyTorch model.
    - test_loader (DataLoader): DataLoader for the test dataset.
    - num_classes (int): Number of classes.
    - threshold (float): Threshold to convert logits to binary predictions.

    Returns:
    - per_class_accuracy (list): List of accuracy values for each class.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()  # Set model to evaluation mode
    model.to(device)
    
    # Initialize counters
    correct_per_class = torch.zeros(num_classes).to(device)
    total_per_class = torch.zeros(num_classes).to(device)
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            predictions = (torch.sigmoid(outputs) > threshold).float()  # Apply threshold
            
            # Update counters
            correct_per_class += (predictions == labels).sum(dim=0)
            total_per_class += labels.size(0)
    
    # Calculate accuracy per class
    per_class_accuracy = (correct_per_class / total_per_class).cpu().numpy()
    
    # Print results
    for i, acc in enumerate(per_class_accuracy):
        print(f"Class {i}: Accuracy = {acc:.2f}")
    
    return per_class_accuracy

In [13]:
# Load the trained model
model = CNN5Layer(num_classes=15)
model.load_state_dict(torch.load("../vu2/RL4Auditing/split_data/cnn5layer_multi_label.pth"))

# Evaluate per-class accuracy
per_class_accuracy = evaluate_per_class_accuracy(model, test_loader, num_classes=15)

Class 0: Accuracy = 1.00
Class 1: Accuracy = 1.00
Class 2: Accuracy = 0.95
Class 3: Accuracy = 0.79
Class 4: Accuracy = 1.00
Class 5: Accuracy = 1.00
Class 6: Accuracy = 0.78
Class 7: Accuracy = 1.00
Class 8: Accuracy = 1.00
Class 9: Accuracy = 0.70
Class 10: Accuracy = 1.00
Class 11: Accuracy = 1.00
Class 12: Accuracy = 0.86
Class 13: Accuracy = 1.00
Class 14: Accuracy = 0.69


In [16]:
# Get the labels for auditing
def get_wrongly_predicted_indices_for_class(model, test_loader, target_class, threshold=0.5):
    """
    Get the indices of samples in the test set that are wrongly predicted for a specific class.

    Parameters:
    - model (nn.Module): The trained PyTorch model.
    - test_loader (DataLoader): DataLoader for the test set.
    - target_class (int): The target class for which to find wrongly predicted samples.
    - threshold (float): Threshold to convert logits to binary predictions (for multi-label classification).

    Returns:
    - wrong_indices (list): List of dataset indices of samples wrongly predicted for the target class.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    model.to(device)

    wrong_indices = []  # List to store indices of wrongly predicted samples
    dataset = test_loader.dataset  # Access the underlying dataset
    start_idx = 0  # Keep track of dataset indices across batches

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            predictions = (torch.sigmoid(outputs) > threshold).float()  # Convert logits to binary predictions

            # Compare predictions and ground truth for the target class
            target_predictions = predictions[:, target_class]  # Predicted values for the target class
            target_labels = labels[:, target_class]  # True labels for the target class

            incorrect = (target_predictions != target_labels).cpu()  # Identify incorrect predictions for the target class
            batch_indices = torch.arange(start_idx, start_idx + len(images))  # Dataset indices of the current batch
            wrong_indices.extend(batch_indices[incorrect].tolist())  # Add indices of wrongly predicted samples

            # Update start index for the next batch
            start_idx += len(images)
    
    return wrong_indices

In [19]:
model.eval()

# Specify the class of interest
target_class = 3  # Change this to the desired class index

# Get indices of wrongly predicted samples for the target class
wrong_indices = get_wrongly_predicted_indices_for_class(model, test_loader, target_class, threshold=0.5)
print(len(wrong_indices)/12000)

0.2085
